<a href="https://colab.research.google.com/github/deekshitha29082006/WPT/blob/main/miniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install tensorflow gradio pillow numpy

import random
import numpy as np
import tensorflow as tf
import gradio as gr
from PIL import Image, ImageDraw, ImageOps
from tensorflow.keras import layers, models

# Settings
SIZE = 128
CLASSES = ["Diode", "Resistor", "Capacitor", "LED", "Transistor"]
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Generate circuit-symbol images for training
def make_symbol(label):
    img = Image.new("RGB", (SIZE, SIZE), "white")
    d = ImageDraw.Draw(img)

    # Randomize symbol size and position
    cx = random.randint(61, 67)
    cy = random.randint(61, 67)
    scale = random.uniform(0.8, 1.05)
    w = max(3, int(random.randint(4, 7) * scale))
    black = (15, 15, 15)

    def line(points, width=w):
        d.line(points, fill=black, width=width)

    if label in ["Diode", "LED"]:
        left, tip, bar = cx-38, cx+20, cx+24
        top, bottom = cy-25, cy+25
        # Wires
        line([(cx-55, cy), (left, cy)])
        # Triangle-shaped diode symbol
        d.polygon([(left, top), (left, bottom), (tip, cy)],
                  fill=black)
        # White triangle interior
        d.polygon([(left+5, top+8), (left+5, bottom-8),
                   (tip-8, cy)], fill="white")
        # Vertical bar at the tip
        line([(bar, top-4), (bar, bottom+4)], width=w+1)
        line([(bar, cy), (cx+55, cy)])
        if label == "LED":
            line([(cx-5, cy-30), (cx+8, cy-43)])
            line([(cx+2, cy-45), (cx+8, cy-43), (cx+7, cy-37)])
            line([(cx+7, cy-20), (cx+20, cy-33)])
            line([(cx+14, cy-35), (cx+20, cy-33), (cx+19, cy-27)])

    elif label == "Resistor":
        line([(cx-55, cy), (cx-30, cy)])
        points = [(cx-30, cy)]
        for i in range(1, 9):
            x = cx-30 + i*7
            y = cy + (-13 if i % 2 else 13)
            points.append((x, y))
        points.append((cx+30, cy))
        line(points)
        line([(cx+30, cy), (cx+55, cy)])

    elif label == "Capacitor":
        line([(cx-55, cy), (cx-8, cy)])
        line([(cx-8, cy-28), (cx-8, cy+28)], width=w+1)
        line([(cx+8, cy-28), (cx+8, cy+28)], width=w+1)
        line([(cx+8, cy), (cx+55, cy)])

    elif label == "Transistor":
        # Simple transistor-style symbol
        d.ellipse((cx-22, cy-22, cx+22, cy+22),
                  outline=black, width=w)
        line([(cx-55, cy), (cx-18, cy)])
        line([(cx-5, cy-8), (cx+17, cy-25)])
        line([(cx-5, cy+8), (cx+17, cy+25)])
        line([(cx+17, cy-25), (cx+17, cy-48)])
        line([(cx+17, cy+25), (cx+17, cy+48)])

    # Small position/rotation variation
    angle = random.uniform(-5, 5)
    img = img.rotate(angle, fillcolor="white")
    return img

print("Creating training dataset...")
images, labels = [], []

for class_id, name in enumerate(CLASSES):
    for _ in range(350):
        images.append(np.array(make_symbol(name).convert("L")))
        labels.append(class_id)

X = np.array(images, dtype=np.float32) / 255.0
X = X[..., np.newaxis]
y = np.array(labels, dtype=np.int32)

order = np.random.permutation(len(X))
X, y = X[order], y[order]

# CNN model
model = models.Sequential([
    layers.Input(shape=(SIZE, SIZE, 1)),
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.25),
    layers.Dense(len(CLASSES), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Training CNN model...")
model.fit(
    X, y,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
print("Training completed!")

# Predict uploaded symbol
def recognize(image):
    if image is None:
        return "Please upload a circuit-symbol image.", {}

    image = ImageOps.grayscale(image)
    image = ImageOps.autocontrast(image)
    image = ImageOps.pad(image, (SIZE, SIZE), color=255)
    arr = np.array(image, dtype=np.float32) / 255.0
    arr = arr.reshape(1, SIZE, SIZE, 1)

    probabilities = model.predict(arr, verbose=0)[0]
    best = int(np.argmax(probabilities))

    result = (
        f"Predicted symbol: {CLASSES[best]}\n"
        f"Model confidence: {probabilities[best]*100:.2f}%\n\n"
        "Educational demo: predictions on unfamiliar symbols "
        "may be incorrect."
    )
    scores = {
        CLASSES[i]: float(probabilities[i])
        for i in range(len(CLASSES))
    }
    return result, scores

# Phone-friendly Gradio interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Electronic Component Symbol Recognition")
    gr.Markdown(
        "Upload a clear circuit symbol, then press Recognize Symbol."
    )
    image_input = gr.Image(type="pil", label="Upload symbol image")
    predict_btn = gr.Button("Recognize Symbol", variant="primary")
    output = gr.Textbox(label="Prediction", lines=4)
    scores = gr.Label(label="Class probabilities")

    predict_btn.click(
        fn=recognize,
        inputs=image_input,
        outputs=[output, scores]
    )

demo.launch(share=True)